In [40]:
import pandas as pd

df = pd.read_csv("btc_1h.csv", index_col=0, parse_dates=True)# ucitavanje podataka

df.index = pd.to_datetime(df.index)
df = df.asfreq('h')  # 'H' = hourly, 'D' = daily

df.head()

,Open,High,Low,Close,Volume
Datetime,,,,,
2017-08-17 04:00:00,4261.48,4313.62,4261.32,4308.83,47.181009
2017-08-17 05:00:00,4308.83,4328.69,4291.37,4315.32,23.234916
2017-08-17 06:00:00,4330.29,4345.45,4309.37,4324.35,7.229691
2017-08-17 07:00:00,4316.62,4349.99,4287.41,4349.99,4.443249
2017-08-17 08:00:00,4333.32,4377.85,4333.32,4360.69,0.972807


In [41]:
df["price_24h"] = df["Close"].shift(-24)
df = df.dropna()# brisanje jer rolling i lag nekada stvaraju prazne vrednosti.
df.head() #ispis :)

arima_series = df["price_24h"]  # target

# Postavljanje indeksa sa frekvencijom da ARIMA ne baca warning
arima_series.index = pd.date_range(start=df.index[0], periods=len(arima_series), freq='h')

In [42]:
# split: 60% train, 20% val, 20% test
n = len(arima_series)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_series = arima_series[:train_end]
val_series   = arima_series[train_end:val_end]
test_series  = arima_series[val_end:]

print(len(train_series), len(val_series), len(test_series))

44747 14916 14916


#!pip install pmdarima

from pmdarima.arima import auto_arima

# Automatski traži optimalne p,d,q
stepwise_model = auto_arima(
    train_series,
    start_p=0, max_p=10,
    start_q=0, max_q=10,
    max_d=2,
    seasonal=False,
    trace=False,           # da ne prikazuje iteracije
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

# Sačuvamo optimalne p,d,q u promenljivu
optimal_order = stepwise_model.order
# Ispis da vidimo koje je odabrao brojeve
print("Optimalni (p,d,q):", optimal_order)

In [43]:
#!pip install statsmodels


from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
import numpy as np


# ARIMA model. Funkcija iznad kaze da su optimalni (p,d,q): (4, 1, 4). poziv traje 15min i zato je zakomentarisan
model = ARIMA(train_series, order=(4, 1, 4))  #model = ARIMA(train_series, order=optimal_order)  # postavlja se p,d,q preko funckija optimal_order koja vrace p, d i q
#              Optimalni (p,d,q): (4, 1, 4)
model_fit = model.fit()

preds_val = model_fit.forecast(steps=len(val_series))


# evaluacija
mae = mean_absolute_error(val_series, preds_val)
rmse = np.sqrt(mean_squared_error(val_series, preds_val))
mape = np.mean(np.abs((val_series.values - preds_val)/val_series.values)) * 100
r2 = r2_score(val_series, preds_val)

print(f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | MAPE: {mape:.2f}% | R²: {r2:.4f}")

MAE: 16224.3093 | RMSE: 22681.6253 | MAPE: 36.30% | R²: -0.8735
